In [ ]:
import sys

In [ ]:
helpers_folder = "./helpers"
sys.path.append(helpers_folder)

from plotting_functions import *
from data_processing import *

In [ ]:
pe_plotstyles = pd.read_csv("config/PE_plotstyles.csv", sep=";").set_index("variable")

## REMIND scenario plots

In [ ]:
plotyears = [2005, 2010, 2015, 2020, 2025, 2030, 2035, 2040, 2045, 2050]
plotyears_climate = [2005, 2010, 2015, 2020, 2025, 2030, 2035, 2040, 2045, 2050, 2055, 2060, 2070, 2080, 2090, 2100]

In [ ]:
scenfolder = "../data/remind_runs"

dflist = []
for fp in os.listdir(scenfolder):
    if fp.endswith("_feedstock.mif"):
        df = pd.read_csv(os.path.join(scenfolder, fp), sep=";")
        df = df[df["Region"] == "World"]
        cols = ["Scenario", "Variable"] + [str(y) for y in plotyears_climate]
        df = df[cols].set_index(["Scenario", "Variable"]).melt(
            ignore_index=False, var_name="Year", value_name="Value").reset_index()
        df["Year"] = df["Year"].astype(int)
        dflist.append(df)

scendata = pd.concat(dflist, ignore_index=True)
scenvariables = scendata["Variable"].unique()

### 

In [ ]:
some_scens = ["SSP2-NPi2025", "SSP2-PkBudg750", "SSP2-PkBudg750-lowbio100EJ-1stgenlim0p6EJ"]
labels = ["CP", "NZ", "NZ-SCI"]
colors = ["gray", "C0", "C2"]
lbl2color = dict(zip(labels, colors))

### Drivers

In [ ]:
fig, axs = plt.subplots(nrows=1, ncols=2, sharex=True, sharey=False, figsize=(7, 4))

ax = axs[0]
ax.set_title("Population")
pdata = scendata.loc[scendata["Variable"] == "Population"].pivot(
    index="Year", columns="Scenario", values="Value"
)[some_scens]
pdata = pdata[pdata.index.isin(plotyears)]
pdata.columns = labels
pdata = pdata / 1e03
ax.set_ylabel("Population [billion]")
pdata.plot.line(ax=ax, color=lbl2color)
ax.grid()

ax = axs[1]
ax.set_title("GDP")
pdata = scendata.loc[scendata["Variable"] == "GDP|PPP"].pivot(
    index="Year", columns="Scenario", values="Value"
)[some_scens]
pdata = pdata[pdata.index.isin(plotyears)]
pdata = pdata / 1e03
pdata.columns = labels
ax.set_ylabel("GDP [trillion USD2017]")
pdata.plot.line(ax=ax, color=lbl2color)
ax.grid()

fig.tight_layout()


### Emissions and warming

In [ ]:
fig, axs = plt.subplots(nrows=1, ncols=2, sharex=True, sharey=False, figsize=(8, 5))

ax = axs[0]
ax.set_title("Global CO2 emissions")
pdata = scendata.loc[scendata["Variable"] == "Emi|CO2"].pivot(index="Year", columns="Scenario", values="Value")[some_scens]
pdata.columns = labels
pdata = pdata / 1e03
ax.set_ylabel("Emissions (Gt CO2 / yr)")
ax.axhline(y=0, alpha=0.7, color="black")
pdata.plot.line(ax=ax, color=lbl2color)
ax.set_xticks([2005, 2025, 2050, 2100])
ax.grid()

tempvars = [f"MAGICC7 AR6|Surface Temperature (GSAT)|{p:.1f}th Percentile" for p in [25, 50, 75]]
ax = axs[1]
ax.set_title("Global warming")
pdata = scendata.loc[scendata["Variable"].isin(tempvars)].pivot(index=["Variable", "Year"], columns="Scenario", values="Value")[some_scens]
ax.set_ylabel("GSAT (°C)")
pdata.columns = labels
pdata.loc[tempvars[1]].plot.line(ax=ax, color=lbl2color)
ax.set_xticks([2005, 2025, 2050, 2100])
ax.grid()
for i, scen in enumerate(pdata.columns):
    color = lbl2color[scen]
    lower = pdata.loc[tempvars[0]][scen]
    upper = pdata.loc[tempvars[2]][scen]
    ax.fill_between(lower.index, lower.values, upper.values, color=color, alpha=0.2)

fig.tight_layout()

### Final energy demands

In [ ]:
FEvars = {
    "Buildings": "FE|++|Buildings",
    "Industry": "FE|++|Industry",
    "Transport - Pass": "FE|Transport|Pass",
    "Transport - Freight": "FE|Transport|Freight",
    "CDR": "FE|++|CDR",
}
FEvar2color = {v: basecolors[k] for k, v in FEvars.items()}

In [ ]:
scendata[scendata["Variable"] == "FE"].pivot(index="Year", columns="Scenario", values="Value").loc[2050][some_scens]

In [ ]:
pdata = scendata[scendata["Variable"].isin(FEvars.values())].copy()
pdata = pdata[pdata["Year"].isin(plotyears)]
some_scens = ["SSP2-NPi2025", "SSP2-PkBudg750", "SSP2-PkBudg750-lowbio100EJ-1stgenlim0p6EJ"]

fig, axs = plt.subplots(1, 3, figsize=(6, 4), sharey=True)

for i, scen in enumerate(some_scens):
    ax = axs.flat[i]
    sel = pdata[pdata["Scenario"] == scen].copy()
    pivot = sel.pivot(index="Year", columns="Variable", values="Value")[FEvars.values()]
    pivot.plot.area(ax=ax, label=scen, color=FEvar2color, legend=False)
    ax.set_title(labels[i])
    ax.set_xticks([2005, 2025, 2050])
    ax.grid()

handles = []
for k, v in FEvars.items():
    c = FEvar2color[v]
    handles.append(Patch(color=c, label=k))
fig.legend(handles=handles, loc="center left", bbox_to_anchor=(1., 0.5))

fig.supylabel("Final energy (EJ/yr)")
fig.tight_layout()


## Primary energy production

In [ ]:
PEvars = pe_plotstyles.index
colors = pe_plotstyles["color"].to_dict()

pdata = scendata[scendata["Variable"].isin(PEvars)].copy()
pdata = pdata[pdata["Year"].isin(plotyears)]
some_scens = ["SSP2-NPi2025", "SSP2-PkBudg750", "SSP2-PkBudg750-lowbio100EJ-1stgenlim0p6EJ"]

fig, axs = plt.subplots(1, 3, figsize=(6, 4), sharey=True)

for i, scen in enumerate(some_scens):
    ax = axs.flat[i]
    sel = pdata[pdata["Scenario"] == scen].copy()
    pivot = sel.pivot(index="Year", columns="Variable", values="Value")[PEvars]
    pivot.plot.area(ax=ax, label=scen, color=colors, legend=False)
    ax.set_title(labels[i])
    ax.set_xticks([2005, 2025, 2050])
    ax.grid()

handles = []
for v in PEvars:
    c = colors[v]
    lbl = pe_plotstyles.loc[v, "label"]
    handles.append(Patch(color=c, label=lbl))
fig.legend(handles=handles, loc="center left", bbox_to_anchor=(1., 0.5))

fig.supylabel("Primary energy (EJ/yr)")
fig.tight_layout()


### Change from NZ to NZ-SCI

In [ ]:
sel = pdata[pdata["Year"] >= 2030].set_index(["Scenario", "Variable", "Year"])["Value"]
diff = sel.loc["SSP2-PkBudg750-lowbio100EJ-1stgenlim0p6EJ"] - sel.loc["SSP2-PkBudg750"]
diff = diff.unstack("Variable")

fig = plt.figure(figsize=(6, 4))
ax = plt.gca()
diff.plot.bar(stacked=True, ax=ax, color=colors, legend=False)
ax.axhline(y=0, color="black", alpha=0.7)
plt.grid()
ax.set_ylabel("Difference in primary energy (EJ/yr)")
fig.suptitle("Difference between NZ-SCI and NZ scenarios")

handles = []
for v in PEvars:
    c = colors[v]
    lbl = pe_plotstyles.loc[v, "label"]
    handles.append(Patch(color=c, label=lbl))
fig.legend(handles=handles, loc="center left", bbox_to_anchor=(0.9, 0.5))


## Biomass production

In [ ]:
bioprod_colors = {
    'PE|Production|Biomass|+|1st Generation': '#ffcc00',
    'PE|Production|Biomass|Lignocellulosic|+|Energy Crops': '#005900',
    'PE|Production|Biomass|Lignocellulosic|+|Residues': "#aa6204"
}

In [ ]:
biovars = list(bioprod_colors.keys())
pdata = scendata[scendata["Variable"].isin(biovars)].copy()
pdata = pdata[pdata["Year"].isin(plotyears)]

fig, axs = plt.subplots(1, 3, figsize=(6, 4), sharey=True)

for i, scen in enumerate(some_scens):
    ax = axs.flat[i]
    sel = pdata[pdata["Scenario"] == scen].copy()
    pivot = sel.pivot(index="Year", columns="Variable", values="Value")[biovars]
    pivot.plot.area(ax=ax, label=scen, color=bioprod_colors, legend=False)
    ax.set_title(labels[i])
    ax.set_xticks([2005, 2025, 2050])
    ax.grid()

handles = []
for v, c in bioprod_colors.items():
    lbl = v.split("|")[-1]
    handles.append(Patch(color=c, label=lbl))
fig.legend(handles=handles, loc="center left", bbox_to_anchor=(1., 0.5))

fig.supylabel("Biomass production (EJ/yr)")
fig.tight_layout()


In [ ]:
pdata.set_index(["Scenario", "Year", "Variable"]).loc[
    :, 2050, "PE|Production|Biomass|Lignocellulosic|+|Energy Crops"
]

In [ ]:
48.25 - 31.3